# Day 20 — pathway 수준으로 해석하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-3/day-20.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-3/day-20.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 주머니에서 공 꺼내기

In [ ]:
from scipy.stats import hypergeom

N = 600      # 배경: 검정한 유전자 수 (주머니 속 공)
K = 40       # 그중 이 gene set에 속한 유전자 수 (빨간 공)
n = 44       # 후보 유전자 수 (꺼낸 공)
k = 8        # 후보 중 gene set에 속한 수 (꺼낸 빨간 공)

expected = n * K / N
p = hypergeom.sf(k - 1, N, K, n)       # k개 이상 겹칠 확률
print("우연히 기대되는 겹침:", round(expected, 2))
print("fold enrichment:", round(k / expected, 2))
print("p-value:", round(p, 4))

### 예시 2 — 배경을 잘못 잡으면

같은 겹침(후보 44개 중 8개)을 두고, 배경만 "전체 유전자 20,000개, 그중 집합 유전자 300개"로 바꿔 봅니다. 숫자는 설명용입니다.

In [ ]:
p_panel = hypergeom.sf(8 - 1, 600, 40, 44)
p_genome = hypergeom.sf(8 - 1, 20000, 300, 44)
print(f"패널을 배경으로: p = {p_panel:.4f}")
print(f"전체 유전자를 배경으로: p = {p_genome:.1e}")

### 예시 3 — GMT 파일은 한 줄이 gene set 하나다

In [ ]:
line = "PRAC:001\tinflammation (fake)\tPrac01\tPrac02\tPrac03"
parts = line.split("\t")
print("ID:", parts[0])
print("이름:", parts[1])
print("유전자:", parts[2:])

## 직접 해보기

1. 설정 셀. 실제 모드에서는 gene set 파일 경로를 직접 넣어야 합니다.

In [ ]:
# 이 셀은 Day 15–18의 산출물 파일이 필요합니다. Colab은 노트북마다 런타임이 따로라서,
# 그 파일을 이 런타임에 업로드하거나 Google Drive에 두고 경로를 맞추세요.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

MODE = "practice"          # 연습이 끝나면 "real"로 바꾸세요
RAW_DIR = Path("../data/osdr_536_practice" if MODE == "practice" else "../data/osdr_536")
OUT_DIR = RAW_DIR / "processed"
FIG_DIR = Path("../figures")

GMT_PATH = None            # 실제 모드: 여기에 실제 GMT 파일 경로를 넣으세요 (아래 2번 참고)
PREFIXES = ("GO:", "REAC:")    # 사용할 gene set ID의 접두어. 파일을 열어 본 뒤 정하세요

de = pd.read_csv(OUT_DIR / "de_table.csv")
background = set(de["gene"])
hits = {
    "all p<0.05": set(de.loc[de["p_value"] < 0.05, "gene"]),
    "FLT down": set(de.loc[(de["p_value"] < 0.05) & (de["log2FC"] < 0), "gene"]),
    "FLT up": set(de.loc[(de["p_value"] < 0.05) & (de["log2FC"] > 0), "gene"]),
}
print("배경:", len(background), "|", {k: len(v) for k, v in hits.items()})

- 한 가지 출처: g:Profiler가 배포하는 마우스 통합 GMT. 2026-11-12에 `https://biit.cs.ut.ee/gprofiler/static/gprofiler_full_mmusculus.name.gmt` 주소에서 약 19MB 파일이 내려받아지고, 한 줄이 "ID, 이름, 유전자 기호들" 구조이며 ID 접두어가 `GO`, `REAC`, `WP`, `HP`, `CORUM`인 것을 확인했습니다. 주소와 내용은 바뀔 수 있으니 g:Profiler 사이트의 데이터 출처 안내에서 현재 버전과 이용 조건을 직접 확인하세요.
   - 받은 파일의 출처, 날짜, 버전을 Day 15처럼 `data/README.md`에 추가합니다.

In [ ]:
if MODE == "practice":
    GMT_PATH = RAW_DIR / "practice_sets.gmt"
    fake_sets = [
        ("PRAC:001", "inflammation (fake)", range(1, 11)),
        ("PRAC:002", "oxidative stress (fake)", range(11, 21)),
        ("PRAC:003", "synapse (fake)", range(21, 31)),
        ("PRAC:004", "myelin (fake)", range(31, 41)),
        ("PRAC:005", "apoptosis (fake)", range(41, 51)),
        ("PRAC:006", "mixed (fake)", range(5, 60, 6)),
    ]
    with open(GMT_PATH, "w", encoding="utf-8") as f:
        for set_id, name, idx in fake_sets:
            f.write("\t".join([set_id, name] + [f"Prac{i:02d}" for i in idx] + ["NotInPanel1"]) + "\n")
    PREFIXES = ("PRAC:",)

assert GMT_PATH is not None, "실제 모드에서는 1번 셀의 GMT_PATH에 파일 경로를 넣어야 합니다."

def read_gmt(path):
    """GMT 파일을 {ID: (이름, 유전자 집합)} 딕셔너리로 읽는다."""
    sets = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                sets[parts[0]] = (parts[1], set(parts[2:]))
    return sets

gene_sets = {k: v for k, v in read_gmt(GMT_PATH).items() if k.startswith(PREFIXES)}
first_id = next(iter(gene_sets))
print("읽은 gene set 수:", len(gene_sets))
print("첫 번째:", first_id, "|", gene_sets[first_id][0], "|", sorted(gene_sets[first_id][1])[:3])

3. 파일의 유전자 이름이 우리 데이터의 유전자 기호와 같은 체계인지 확인합니다. 겹치는 비율이 낮으면 종(species)이나 ID 체계가 다른 파일입니다.

In [ ]:
all_set_genes = set().union(*(genes for _, genes in gene_sets.values()))
covered = background & all_set_genes
print(f"배경 유전자 {len(background)}개 중 gene set에 한 번이라도 나오는 것: {len(covered)}개")

4. ORA 함수를 만들고 세 가지 후보 목록에 대해 실행합니다. 배경 안에서 유전자가 5개 미만이거나 배경의 절반을 넘는 집합은 평가하지 않습니다.

In [ ]:
def run_ora(hit_genes, gene_sets, background, min_size=5):
    rows = []
    N, n = len(background), len(hit_genes)
    for set_id, (name, genes) in gene_sets.items():
        in_bg = genes & background                 # 배경(검정한 유전자) 안으로 제한
        K = len(in_bg)
        if K < min_size or K > N / 2:
            continue
        overlap = in_bg & hit_genes
        k = len(overlap)
        expected = n * K / N
        rows.append({
            "set_id": set_id, "name": name, "set_size_in_background": K,
            "overlap": k, "expected": round(expected, 2),
            "fold_enrichment": round(k / expected, 2) if expected > 0 else np.nan,
            "p_value": hypergeom.sf(k - 1, N, K, n),
            "genes": ", ".join(sorted(overlap)),
        })
    out = pd.DataFrame(rows)
    out["FDR"] = multipletests(out["p_value"], method="fdr_bh")[1]
    return out.sort_values("p_value").reset_index(drop=True)

tables = []
for label, hit_genes in hits.items():
    if len(hit_genes) > 0:
        tables.append(run_ora(hit_genes, gene_sets, background).assign(hit_list=label))
pathway = pd.concat(tables, ignore_index=True)

print("평가한 gene set 수(목록당):", pathway["set_id"].nunique())
print("FDR < 0.05 인 행:", int((pathway["FDR"] < 0.05).sum()))
show = ["hit_list", "name", "set_size_in_background", "overlap", "fold_enrichment", "p_value", "FDR"]
print(pathway.sort_values("p_value")[show].head(6).round(4).to_string(index=False))
pathway.round(5).to_csv(OUT_DIR / "pathway_table.csv", index=False)

5. 방향별(FLT down / FLT up) 상위 집합을 dotplot으로 그립니다.

In [ ]:
top = (pathway[pathway["hit_list"] != "all p<0.05"]
       .query("overlap >= 2")
       .sort_values("p_value")
       .groupby("hit_list").head(8))
top = top.assign(label=top["name"].str.slice(0, 45) + " [" + top["hit_list"] + "]").iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 0.35 * len(top) + 1.8))
sc = ax.scatter(top["fold_enrichment"], top["label"], s=top["overlap"] * 40,
                c=-np.log10(top["p_value"]), cmap="viridis")
ax.axvline(1, linestyle="--", linewidth=0.8, color="grey")
ax.set_xlabel("Fold enrichment (observed / expected overlap)")
ax.set_title(f"ORA of genes with unadjusted p < 0.05 (exploratory) [data: {MODE}]\nbackground = tested panel genes; dot size = overlap", fontsize=9)
fig.colorbar(sc, ax=ax, label="-log10(p-value)")
fig.tight_layout()
fig.savefig(FIG_DIR / f"osdr_{MODE}_pathway_dotplot.png", dpi=150)
print("그림에 넣은 gene set 수:", len(top))

8. (선택) 웹 도구로 교차 확인합니다. g:Profiler 같은 enrichment 웹 도구에 후보 유전자를 붙여 넣되, 종을 *Mus musculus*로 고르고 **custom background**에 검정한 유전자 전체를 넣습니다. 내려받은 결과 파일의 열 이름은 도구와 버전마다 다르므로 `pd.read_csv("여기에 실제 파일명을 넣으세요").columns.tolist()`로 먼저 확인한 뒤 사용합니다. 붙여 넣을 목록은 아래처럼 만듭니다.

In [ ]:
(OUT_DIR / "genes_background.txt").write_text("\n".join(sorted(background)))
(OUT_DIR / "genes_flt_down.txt").write_text("\n".join(sorted(hits["FLT down"])))
(OUT_DIR / "genes_flt_up.txt").write_text("\n".join(sorted(hits["FLT up"])))
print("목록 파일 3개 저장")